# Notebook 03 — Analysis, Insights & Machine Learning

**Goal:** Extract analytical insights from the silver layer and train a 
machine learning model to predict tip amounts.

**Analyses performed:**
1. Trip volume patterns: monthly, daily, hourly
2. Revenue and fare analysis by time of day and trip distance
3. Top pickup and dropoff zones
4. Tipping behaviour analysis
5. Payment type breakdown
6. ML: Linear Regression model predicting tip amount

**Outputs:** Aggregated tables written to gold layer for Power BI dashboards

In [0]:
# ─────────────────────────────────────────────────────────────
# Setup
# ─────────────────────────────────────────────────────────────
from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, round as spark_round,
    desc, asc, when, year, month, hour
)

storage_account_name = "staxidatakc250253104"
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net"
gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net"
bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net"

# Load silver data
df = spark.read.format("delta").load(f"{silver_path}/yellow_taxi_2023")

# Register as a temporary SQL view so we can use SQL too
df.createOrReplaceTempView("trips")

print(f"✓ Loaded {df.count():,} trips from silver layer")
print(f"✓ Registered as SQL view 'trips'")

✓ Loaded 35,575,601 trips from silver layer
✓ Registered as SQL view 'trips'


In [0]:
# ─── Analysis 1: Trip volume and revenue by month ──────────
monthly_summary = spark.sql("""
    SELECT 
        pickup_month_num AS month,
        COUNT(*) AS total_trips,
        ROUND(SUM(total_amount), 2) AS total_revenue,
        ROUND(AVG(total_amount), 2) AS avg_fare,
        ROUND(AVG(trip_distance), 2) AS avg_distance_miles,
        ROUND(AVG(trip_duration_min), 2) AS avg_duration_min,
        ROUND(AVG(tip_pct), 2) AS avg_tip_pct
    FROM trips
    GROUP BY pickup_month_num
    ORDER BY month
""")

display(monthly_summary)

month,total_trips,total_revenue,avg_fare,avg_distance_miles,avg_duration_min,avg_tip_pct
1,2881181,7.879224706E7,27.35,3.42,14.51,21.19
2,2729715,7.446452365E7,27.28,3.34,14.79,20.78
3,3187161,8.987459521E7,28.2,3.45,15.64,20.73
4,3074422,8.816315706E7,28.68,3.55,16.01,20.75
5,3278431,9.623415786E7,29.35,3.57,16.96,20.65
6,3081849,9.038342323E7,29.33,3.6,16.66,21.46
7,2710192,7.874638586E7,29.06,3.65,16.02,22.42
8,2626232,7.659744464E7,29.17,3.68,16.1,20.33
9,2602639,7.852785529E7,30.17,3.59,17.93,20.22
10,3240989,9.638307463E7,29.74,3.52,17.45,21.14


In [0]:
# ─── Analysis 2: Hourly trip patterns (rush hour identification) ───
hourly_pattern = spark.sql("""
    SELECT 
        pickup_hour AS hour,
        COUNT(*) AS trip_count,
        ROUND(AVG(fare_amount), 2) AS avg_fare,
        ROUND(AVG(trip_duration_min), 2) AS avg_duration,
        ROUND(AVG(tip_pct), 2) AS avg_tip_pct
    FROM trips
    GROUP BY pickup_hour
    ORDER BY pickup_hour
""")

display(hourly_pattern)

hour,trip_count,avg_fare,avg_duration,avg_tip_pct
0,995086,20.13,14.18,20.62
1,665094,18.19,12.9,21.9
2,436570,17.04,12.05,26.98
3,283781,18.02,12.08,20.11
4,181273,23.89,13.89,18.9
5,194766,27.95,15.42,17.55
6,475533,23.24,15.31,21.96
7,946648,19.53,15.15,21.25
8,1321874,18.61,15.56,20.04
9,1514795,18.78,15.82,19.71


In [0]:
# ─── Analysis 3: Day of week patterns ───────────────────────
dow_pattern = spark.sql("""
    SELECT 
        pickup_day_name AS day_of_week,
        pickup_day_of_week AS day_num,
        COUNT(*) AS trip_count,
        ROUND(AVG(fare_amount), 2) AS avg_fare,
        ROUND(AVG(tip_pct), 2) AS avg_tip_pct,
        ROUND(SUM(total_amount), 2) AS total_revenue
    FROM trips
    GROUP BY pickup_day_name, pickup_day_of_week
    ORDER BY pickup_day_of_week
""")

display(dow_pattern)

day_of_week,day_num,trip_count,avg_fare,avg_tip_pct,total_revenue
Sunday,1,4460109,20.63,20.19,1.31972097E8
Monday,2,4444384,20.42,20.53,1.3334380985E8
Tuesday,3,5170887,19.57,21.7,1.494153305E8
Wednesday,4,5474980,19.54,21.18,1.5825673018E8
Thursday,5,5586273,19.8,21.95,1.6328878901E8
Friday,6,5277074,19.54,20.66,1.5248138832E8
Saturday,7,5161894,18.75,20.29,1.3934273486E8


In [0]:
# ─── Analysis 4: Top 15 busiest pickup zones ───────────────
top_pickup_zones = spark.sql("""
    SELECT 
        PULocationID AS pickup_zone_id,
        COUNT(*) AS pickup_count,
        ROUND(AVG(fare_amount), 2) AS avg_fare,
        ROUND(AVG(trip_distance), 2) AS avg_distance,
        ROUND(SUM(total_amount), 2) AS total_revenue
    FROM trips
    GROUP BY PULocationID
    ORDER BY pickup_count DESC
    LIMIT 15
""")

display(top_pickup_zones)

pickup_zone_id,pickup_count,avg_fare,avg_distance,total_revenue
132.0,1888115,63.65,16.05,1.5349581262E8
237.0,1701192,13.13,1.81,3.51748366E7
161.0,1674847,16.68,2.48,4.240802941E7
236.0,1500390,13.61,1.98,3.182689812E7
162.0,1290155,16.06,2.41,3.160629349E7
138.0,1262227,43.53,9.71,8.469380904E7
186.0,1244844,17.23,2.4,3.155454982E7
230.0,1197007,19.55,3.25,3.450337519E7
142.0,1180517,14.38,2.22,2.624822189E7
170.0,1066329,15.97,2.37,2.589379135E7


In [0]:
# ─── Analysis 5: Tipping behaviour by payment type and time ─
tip_analysis = spark.sql("""
    SELECT 
        time_of_day,
        trip_distance_bucket,
        COUNT(*) AS trip_count,
        ROUND(AVG(fare_amount), 2) AS avg_fare,
        ROUND(AVG(tip_amount), 2) AS avg_tip,
        ROUND(AVG(tip_pct), 2) AS avg_tip_pct
    FROM trips
    WHERE payment_type = 1   -- credit card only (cash tips often not recorded)
    GROUP BY time_of_day, trip_distance_bucket
    ORDER BY time_of_day, trip_distance_bucket
""")

display(tip_analysis)

time_of_day,trip_distance_bucket,trip_count,avg_fare,avg_tip,avg_tip_pct
Afternoon,Long (3-10 mi),1474224,30.1,6.36,21.07
Afternoon,Medium (1-3 mi),4202830,14.23,3.35,24.18
Afternoon,Short (<1 mi),1865948,8.31,2.34,30.19
Afternoon,Very Long (10+ mi),826470,66.34,13.1,24.37
Evening,Long (3-10 mi),1470751,27.61,6.02,22.14
Evening,Medium (1-3 mi),4053453,13.35,3.36,26.19
Evening,Short (<1 mi),1620449,7.82,2.44,33.4
Evening,Very Long (10+ mi),551042,66.15,13.04,20.26
Morning,Long (3-10 mi),1124700,28.19,5.72,20.4
Morning,Medium (1-3 mi),3208962,13.26,3.08,23.86


In [0]:
# ─── Analysis 6: Payment type breakdown ─────────────────────
payment_mix = spark.sql("""
    SELECT 
        CASE payment_type
            WHEN 1 THEN 'Credit card'
            WHEN 2 THEN 'Cash'
            WHEN 3 THEN 'No charge'
            WHEN 4 THEN 'Dispute'
            WHEN 5 THEN 'Unknown'
            WHEN 6 THEN 'Voided trip'
            ELSE 'Other'
        END AS payment_method,
        COUNT(*) AS trip_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct_of_trips,
        ROUND(AVG(total_amount), 2) AS avg_total,
        ROUND(AVG(tip_pct), 2) AS avg_tip_pct
    FROM trips
    GROUP BY payment_type
    ORDER BY trip_count DESC
""")

display(payment_mix)

payment_method,trip_count,pct_of_trips,avg_total,avg_tip_pct
Credit card,29122780,81.86,29.72,25.61
Cash,6085232,17.11,25.19,0.0
Dispute,242928,0.68,25.82,0.15
No charge,124661,0.35,23.49,0.1


In [0]:
# ─── Write all aggregates to gold layer ─────────────────────
print("Writing gold layer aggregates...")

monthly_summary.write.format("delta").mode("overwrite").save(f"{gold_path}/monthly_summary")
print("  ✓ monthly_summary")

hourly_pattern.write.format("delta").mode("overwrite").save(f"{gold_path}/hourly_pattern")
print("  ✓ hourly_pattern")

dow_pattern.write.format("delta").mode("overwrite").save(f"{gold_path}/dow_pattern")
print("  ✓ dow_pattern")

top_pickup_zones.write.format("delta").mode("overwrite").save(f"{gold_path}/top_pickup_zones")
print("  ✓ top_pickup_zones")

tip_analysis.write.format("delta").mode("overwrite").save(f"{gold_path}/tip_analysis")
print("  ✓ tip_analysis")

payment_mix.write.format("delta").mode("overwrite").save(f"{gold_path}/payment_mix")
print("  ✓ payment_mix")

print("\n✓ All gold tables written")

Writing gold layer aggregates...
  ✓ monthly_summary
  ✓ hourly_pattern
  ✓ dow_pattern
  ✓ top_pickup_zones
  ✓ tip_analysis
  ✓ payment_mix

✓ All gold tables written


## Machine Learning — Tip Amount Prediction

A linear regression model is trained using Spark MLlib to predict `tip_amount` 
from trip characteristics. This serves as an example of ML-based insight extraction 
on the curated silver layer.

**Features:** trip_distance, trip_duration_min, passenger_count, fare_amount, 
pickup_hour, pickup_day_of_week  
**Target:** tip_amount  
**Algorithm:** Linear Regression (Spark MLlib)  
**Train/Test split:** 80/20

In [0]:
# ─────────────────────────────────────────────────────────────
# Prepare features for ML
# We use only credit card trips (payment_type = 1) because
# cash tips are not reliably recorded in the TLC data
# ─────────────────────────────────────────────────────────────

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

ml_df = (
    df
    .filter(col("payment_type") == 1)                # credit cards only
    .filter(col("tip_amount") >= 0)
    .filter(col("tip_amount") < 100)                 # remove tip outliers
    .select(
        "trip_distance",
        "trip_duration_min",
        "passenger_count",
        "fare_amount",
        "pickup_hour",
        "pickup_day_of_week",
        "tip_amount"
    )
    .dropna()
)

print(f"✓ ML dataset prepared: {ml_df.count():,} records")
display(ml_df.limit(5))

✓ ML dataset prepared: 29,122,332 records


trip_distance,trip_duration_min,passenger_count,fare_amount,pickup_hour,pickup_day_of_week,tip_amount
10.37,62.72,1.0,58.3,9,3,15.62
1.17,24.23,1.0,19.8,11,3,4.96
16.96,52.0,1.0,70.0,13,3,16.46
0.77,11.83,1.0,10.7,14,3,3.14
1.25,16.4,1.0,14.9,14,3,3.98


In [0]:
# ─── Split ───────────────────────────────────────────────────
train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)
print(f"  Training set: {train_df.count():,} records")
print(f"  Test set:     {test_df.count():,} records")

# ─── Feature vector assembly ─────────────────────────────────
feature_cols = [
    "trip_distance", "trip_duration_min", "passenger_count",
    "fare_amount", "pickup_hour", "pickup_day_of_week"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

print(f"\n✓ Features: {feature_cols}")
print(f"✓ Target:   tip_amount")

  Training set: 23,297,102 records
  Test set:     5,825,230 records

✓ Features: ['trip_distance', 'trip_duration_min', 'passenger_count', 'fare_amount', 'pickup_hour', 'pickup_day_of_week']
✓ Target:   tip_amount


In [0]:
# ─── Train Linear Regression ────────────────────────────────
lr = LinearRegression(
    featuresCol="features",
    labelCol="tip_amount",
    maxIter=10,
    regParam=0.1,
    elasticNetParam=0.5
)

# Build a pipeline (assembler + model)
pipeline = Pipeline(stages=[assembler, lr])

print("Training Linear Regression model...")
model = pipeline.fit(train_df)
print("✓ Training complete")

Training Linear Regression model...
✓ Training complete


In [0]:
# Get the trained linear regression stage (stage 1 of the pipeline)
lr_model = model.stages[-1]

print("MODEL COEFFICIENTS")
print("=" * 60)
print(f"  Intercept: {lr_model.intercept:.4f}\n")
print(f"  {'Feature':<25} {'Coefficient':>15}")
print(f"  {'-'*25} {'-'*15}")
for fname, coef in zip(feature_cols, lr_model.coefficients):
    print(f"  {fname:<25} {coef:>15.6f}")

print(f"\n  Training R²:    {lr_model.summary.r2:.4f}")
print(f"  Training RMSE:  {lr_model.summary.rootMeanSquaredError:.4f}")

MODEL COEFFICIENTS
  Intercept: 1.0875

  Feature                       Coefficient
  ------------------------- ---------------
  trip_distance                    0.162211
  trip_duration_min                0.014665
  passenger_count                  0.000000
  fare_amount                      0.121119
  pickup_hour                      0.007791
  pickup_day_of_week               0.000000

  Training R²:    0.5883
  Training RMSE:  2.5653


In [0]:
# ─── Predict on test set ─────────────────────────────────────
predictions = model.transform(test_df)

# Show a few sample predictions
print("Sample predictions:")
display(predictions.select(
    "trip_distance", "trip_duration_min", "fare_amount",
    "tip_amount", "prediction"
).limit(10))

Sample predictions:


trip_distance,trip_duration_min,fare_amount,tip_amount,prediction
0.01,0.03,114.0,0.0,15.029554535884987
0.01,0.03,3.0,1.65,1.6243305293474852
0.01,0.05,57.8,16.26,8.246348158254882
0.01,0.05,90.0,0.05,12.130789342771859
0.01,0.05,85.0,17.3,11.501823194578083
0.01,0.05,90.0,18.3,12.115207653292408
0.01,0.05,110.0,22.2,14.59211802236889
0.01,0.07,3.0,60.0,1.4924727872226007
0.01,0.07,15.0,1.0,2.992642529199997
0.01,0.07,40.0,8.2,5.973865530634645


In [0]:
# ─── Calculate metrics on test set ───────────────────────────
evaluator_rmse = RegressionEvaluator(
    labelCol="tip_amount", predictionCol="prediction", metricName="rmse"
)
evaluator_r2 = RegressionEvaluator(
    labelCol="tip_amount", predictionCol="prediction", metricName="r2"
)
evaluator_mae = RegressionEvaluator(
    labelCol="tip_amount", predictionCol="prediction", metricName="mae"
)

rmse = evaluator_rmse.evaluate(predictions)
r2   = evaluator_r2.evaluate(predictions)
mae  = evaluator_mae.evaluate(predictions)

print("TEST SET PERFORMANCE")
print("=" * 50)
print(f"  RMSE:  {rmse:.4f}   (root mean squared error in $)")
print(f"  MAE:   {mae:.4f}   (mean absolute error in $)")
print(f"  R²:    {r2:.4f}   (variance explained — higher is better)")

TEST SET PERFORMANCE
  RMSE:  2.5448   (root mean squared error in $)
  MAE:   1.3883   (mean absolute error in $)
  R²:    0.5917   (variance explained — higher is better)


In [0]:
# ─── Save predictions sample to gold for Power BI viz ───────
(
    predictions
    .select("trip_distance", "trip_duration_min", "fare_amount",
            "pickup_hour", "tip_amount", "prediction")
    .sample(fraction=0.01, seed=42)   # 1% sample is plenty for visualisation
    .write
    .format("delta")
    .mode("overwrite")
    .save(f"{gold_path}/ml_predictions_sample")
)

print(f"✓ Predictions sample saved to gold/ml_predictions_sample")

✓ Predictions sample saved to gold/ml_predictions_sample


In [0]:
# Print a clean summary suitable for screenshot/report
print("=" * 60)
print("TIP AMOUNT PREDICTION — MODEL SUMMARY")
print("=" * 60)
print(f"  Algorithm:        Linear Regression (Spark MLlib)")
print(f"  Training records: {train_df.count():,}")
print(f"  Test records:     {test_df.count():,}")
print(f"  Features used:    {len(feature_cols)}")
print()
print(f"  Test R²:    {r2:.4f}")
print(f"  Test RMSE:  ${rmse:.2f}")
print(f"  Test MAE:   ${mae:.2f}")
print()
print(f"  Most influential feature (by coefficient magnitude):")
biggest_feat = max(zip(feature_cols, [abs(c) for c in lr_model.coefficients]), key=lambda x: x[1])
print(f"     {biggest_feat[0]} (coef magnitude: {biggest_feat[1]:.4f})")
print("=" * 60)

TIP AMOUNT PREDICTION — MODEL SUMMARY
  Algorithm:        Linear Regression (Spark MLlib)
  Training records: 23,297,579
  Test records:     5,824,753
  Features used:    6

  Test R²:    0.5917
  Test RMSE:  $2.54
  Test MAE:   $1.39

  Most influential feature (by coefficient magnitude):
     trip_distance (coef magnitude: 0.1622)
